# Pipeline Debug Notebook
逐步測試 daily_update pipeline 的每個環節

In [1]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# 讓 notebook 能 import src/ 的模組
ROOT = Path().resolve().parents[1]
SRC  = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

load_dotenv(ROOT / '.env')
from utils.paths import BrokerPaths
BROKER_ID = '1440'
PATHS = BrokerPaths(BROKER_ID)
print('ROOT:', ROOT)
print('Broker:', BROKER_ID)

ROOT: C:\Users\hcn12\OneDrive\Desktop\NING\Github\BMEM


## 1. FinMind API 連線測試

In [ ]:
from FinMind.data import DataLoader

api = DataLoader()
api.login_by_token(api_token=os.environ['FINMIND_API_KEY'])

TEST_STOCK  = '0050'
TEST_DATE   = '2026-05-14'

# 股價
df_price = api.taiwan_stock_daily(stock_id=TEST_STOCK, start_date='2026-05-11', end_date=TEST_DATE)
print(f'股價 ({TEST_STOCK}):')
display(df_price)

2026-05-14 22:47:43.673 | INFO     | FinMind.data.finmind_api:login_by_token:85 - Login success
2026-05-14 22:47:43.735 | INFO     | FinMind.data.finmind_api:login_by_token:85 - Login success
2026-05-14 22:47:43.736 | INFO     | FinMind.data.finmind_api:get_data:164 - download Dataset.TaiwanStockPrice, data_id: 0050


股價 (0050):


,date,stock_id,Trading_Volume,Trading_money,open,max,min,close,spread,Trading_turnover
0,2026-05-11,0050,96606391,9362479353,96.65,97.15,96.50,96.90,-0.10,148902
1,2026-05-12,0050,102303450,9876959425,96.85,97.35,95.50,96.85,-0.05,142949
2,2026-05-13,0050,130654061,12419149243,94.85,95.80,94.70,95.50,-1.35,260692
3,2026-05-14,0050,57229209,5510647058,96.90,97.20,95.75,96.05,0.55,70419


: 

## 2. 載入本地 Parquet

In [ ]:
from daily_update import (
    _load_all_broker_parquets,
    _load_stock_parquets,
    BROKER_DIR, STOCK_DIR, LOOKBACK_DAYS,
)

TARGET_DATE = '2026-05-14'

cutoff_dt = pd.to_datetime(TARGET_DATE) - pd.Timedelta(days=LOOKBACK_DAYS)
target_dt = pd.to_datetime(TARGET_DATE)

all_broker = _load_all_broker_parquets(BROKER_ID)
combined_broker = all_broker[all_broker['date'] >= cutoff_dt].copy()

print(f'Broker rows: {len(combined_broker):,}')
print(f'Unique stocks: {combined_broker["stock_id"].nunique()}')
print(f'Date range: {combined_broker["date"].min().date()} ~ {combined_broker["date"].max().date()}')
display(combined_broker.tail(3))

In [ ]:
window_stock_ids = combined_broker['stock_id'].astype(str).unique().tolist()
combined_stocks  = _load_stock_parquets(window_stock_ids)
if not combined_stocks.empty:
    combined_stocks['date'] = pd.to_datetime(combined_stocks['date'])
    combined_stocks = combined_stocks[combined_stocks['date'] >= cutoff_dt]

print(f'Stock rows: {len(combined_stocks):,}')
print(f'Date range: {combined_stocks["date"].min().date()} ~ {combined_stocks["date"].max().date()}')
display(combined_stocks.tail(3))

## 3. 計算 Observation Features

In [ ]:
from pipeline_functions import compute_observation_features, FEATURE_COLS

if 'securities_trader_id' not in combined_broker.columns:
    combined_broker['securities_trader_id'] = BROKER_ID

feature_df = compute_observation_features(
    combined_broker, combined_stocks,
    disable_standardize=True,
    contamination_lookback_days=90,
)
feature_df['date'] = pd.to_datetime(feature_df['date'])
valid_df = feature_df.dropna(subset=FEATURE_COLS).copy()

today_valid = valid_df[valid_df['date'] == target_dt]
print(f'valid_df rows: {len(valid_df):,}')
print(f'Today valid candidates: {len(today_valid)}')
display(today_valid[['date','stock_id'] + FEATURE_COLS].head())

## 4. HMM 推論（增量版）

In [ ]:
from pipeline_functions import load_hmm_model, compute_rolling_hmm_proba
from daily_update import HMM_WINDOW

hmm_model = load_hmm_model(str(PATHS.hmm_model_path))
print(f'HMM states: {hmm_model.n_components}')

hmm_input = valid_df.sort_values(['stock_id', 'securities_trader_id', 'date'])

# Phase 1: 全部股票 × 只算今天
today_hmm = compute_rolling_hmm_proba(
    hmm_input, hmm_model,
    feature_cols=FEATURE_COLS, window=HMM_WINDOW,
    inference_dates=[target_dt],
)
print(f'\ntoday_hmm rows: {len(today_hmm)}')
prob_cols = [c for c in today_hmm.columns if c.startswith('prob_S')]
display(today_hmm[['date','stock_id'] + prob_cols].head())

## 5. XGBoost 訊號

In [ ]:
from pipeline_functions import load_xgb_model, generate_signals
from daily_update import LONG_THRESHOLD, SHORT_THRESHOLD, OUTPUT_TOP_N

clf_long  = load_xgb_model(str(PATHS.xgboost_model_path('long')))
clf_short = load_xgb_model(str(PATHS.xgboost_model_path('short')))
XGB_FEATURE_COLS = clf_long.get_booster().feature_names
if not XGB_FEATURE_COLS:
    XGB_FEATURE_COLS = ['z_t', 'c_t', 'a_t', 's_t', 'm_t', 'bias_60d', 'net_buy_amt_60d'] + prob_cols

signals_df = generate_signals(
    today_hmm, clf_long, clf_short,
    feature_cols=XGB_FEATURE_COLS,
    long_threshold=LONG_THRESHOLD,
    short_threshold=SHORT_THRESHOLD,
)

top10 = signals_df.sort_values('pred_prob_long', ascending=False).head(OUTPUT_TOP_N)
print(f'Long  signals (>={LONG_THRESHOLD:.0%}): {signals_df["signal_long"].sum()}')
print(f'Short signals (>={SHORT_THRESHOLD:.0%}): {signals_df["signal_short"].sum()}')
print(f'\nTop-{OUTPUT_TOP_N} Long 候選:')
display(top10[['stock_id','pred_prob_long','pred_prob_short','signal_long','signal_short']])

## 6. 歷史訊號（top-10 × 最近 20 天）

In [ ]:
from daily_update import OUTPUT_HIST_DAYS

top_stocks     = top10['stock_id'].astype(str).tolist()
all_valid_dates = sorted(valid_df['date'].unique())
hist_dates     = [d for d in all_valid_dates if d <= target_dt][-OUTPUT_HIST_DAYS:]

print(f'hist_dates: {pd.to_datetime(hist_dates[0]).date()} ~ {pd.to_datetime(hist_dates[-1]).date()}')

# Phase 2: top-10 × 最近 20 天
hist_input = hmm_input[hmm_input['stock_id'].astype(str).isin(top_stocks)]
hmm_hist   = compute_rolling_hmm_proba(
    hist_input, hmm_model,
    feature_cols=FEATURE_COLS, window=HMM_WINDOW,
    inference_dates=hist_dates, show_progress=False,
)

signals_hist = generate_signals(
    hmm_hist, clf_long, clf_short,
    feature_cols=XGB_FEATURE_COLS,
    long_threshold=LONG_THRESHOLD,
    short_threshold=SHORT_THRESHOLD,
)
print(f'signals_hist rows: {len(signals_hist)}')
display(signals_hist[['date','stock_id','pred_prob_long','pred_prob_short']].head(10))

## 7. 單支股票快速測試

In [ ]:
# 指定一支股票，看完整的 feature + HMM 輸出
INSPECT_SID = top_stocks[0]

sid_features = valid_df[valid_df['stock_id'].astype(str) == INSPECT_SID].sort_values('date')
print(f'{INSPECT_SID} feature rows: {len(sid_features)}')
display(sid_features[['date'] + FEATURE_COLS + ['bias_60d','net_buy_amt_60d']].tail(5))

sid_signals = signals_hist[signals_hist['stock_id'].astype(str) == INSPECT_SID].sort_values('date')
display(sid_signals[['date','pred_prob_long','pred_prob_short','signal_long']].tail(10))

## 8. 跑完整 daily_update（可選）

In [ ]:
# 取消下面的 # 來跑完整 pipeline（會發 email）
# from daily_update import run_daily_update
# run_daily_update(target_date=TARGET_DATE, broker_id=BROKER_ID)